# MSDS 458 — Final Project A.1 Baseline Notebook  
## Do Hidden Layers in CNNs Learn Progressively More Abstract Features?

**Research Question:**  
Do CNN hidden layers learn progressively more abstract visual features as network depth increases, and can we visualize this progression?

**Preliminary Dataset:**  
This notebook uses **MNIST** as a clean, manageable CNN baseline. The goal is to establish the feature-abstraction workflow before optionally extending the same method to the course-provided **9×9 letters/shapes/lines data**.

**Core Comparison:**  
- **Model A:** shallow CNN with 1 convolutional layer  
- **Model B:** deeper CNN with 3 convolutional layers  

**Important 9×9 Architecture Note:**  
The MNIST version can safely use multiple MaxPooling layers because MNIST images are 28×28.  
For a 9×9 course dataset, repeated pooling can collapse the spatial representation too quickly.  
This notebook therefore includes a **small-image-safe Model B option** that reduces pooling when adapting the project to 9×9 inputs.

**Evidence Collected:**  
1. Test accuracy and loss  
2. Training/validation curves  
3. Feature maps across layers  
4. Learned filters  
5. Confusion matrices  
6. Activation sparsity statistics  
7. PCA visualization and silhouette scores for class separability


## Cell 1 — Imports, Setup, and Reproducibility

In [3]:
# Display settings
try:
    from IPython.core.display import display, HTML
    display(HTML("<style>div.output_scroll { height: 60em; }</style>"))
    display(HTML("<style>.container { width:100% !important; }</style>"))
except Exception:
    pass

import os
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout
)
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

# Reproducibility
SEED = 2024
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

# Output folder
OUTPUT_DIR = "topic4_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("TensorFlow version:", tf.__version__)
print("Output folder:", OUTPUT_DIR)


/var/folders/w7/yrqkdw2n2hg91sz77xxy9sjr0000gp/T/ipykernel_83216/218097740.py:3: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


ModuleNotFoundError: No module named 'tensorflow'

## Cell 2 — Load and Preprocess MNIST Data

In [ ]:
# Load MNIST dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

print("Original training images shape:", x_train.shape)
print("Original test images shape:    ", x_test.shape)

# Normalize pixel values from 0-255 to 0-1
x_train = x_train.astype("float32") / 255.0
x_test  = x_test.astype("float32") / 255.0

# Add channel dimension for CNN input: (N, 28, 28) -> (N, 28, 28, 1)
x_train = x_train.reshape(-1, 28, 28, 1)
x_test  = x_test.reshape(-1, 28, 28, 1)

# General settings inferred from the data.
# These make it easier to later adapt the notebook to 9x9 course images.
INPUT_SHAPE = x_train.shape[1:]
NUM_CLASSES = len(np.unique(y_train))

print("\nAfter preprocessing:")
print("x_train:", x_train.shape, x_train.dtype)
print("x_test: ", x_test.shape, x_test.dtype)
print("y_train:", y_train.shape)
print("y_test: ", y_test.shape)
print("INPUT_SHAPE:", INPUT_SHAPE)
print("NUM_CLASSES:", NUM_CLASSES)


## Cell 3 — Visualize Sample Images

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle("Sample MNIST Training Images", fontsize=13, fontweight="bold")

for i, ax in enumerate(axes.flat):
    ax.imshow(x_train[i].reshape(28, 28), cmap="gray")
    ax.set_title(f"Label: {y_train[i]}", fontsize=10)
    ax.axis("off")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "sample_images.png"), dpi=150, bbox_inches="tight")
plt.show()

## Cell 4 — Build Model A: Shallow CNN

Architecture:  
`Input → Conv2D(32) → MaxPool → Flatten → Dense(128) → Dropout → Output`

Model A is the shallow baseline. It is intentionally simple so that its first convolutional feature maps can be compared with the deeper Model B.


In [ ]:
def build_model_A(input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES):
    model = Sequential([
        Input(shape=input_shape, name="input_A"),
        Conv2D(32, kernel_size=(3, 3), activation="relu", padding="same", name="conv_A1"),
        MaxPooling2D(pool_size=(2, 2), name="pool_A1"),
        Flatten(name="flatten_A"),
        Dense(128, activation="relu", name="dense_A1"),
        Dropout(0.30, name="dropout_A"),
        Dense(num_classes, activation="softmax", name="output_A")
    ], name="Model_A_Shallow")

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

model_A = build_model_A()
model_A.summary()


## Cell 5 — Build Model B: Deep CNN

MNIST architecture:  
`Input → Conv2D(32) → MaxPool → Conv2D(64) → MaxPool → Conv2D(128) → MaxPool → Flatten → Dense(256) → Dropout → Output`

**9×9-safe architecture option:**  
For very small 9×9 course images, repeated pooling can collapse spatial dimensions too quickly.  
Set `small_image_safe=True` when adapting this notebook to 9×9 data:

`Input → Conv2D(32) → Conv2D(64) → MaxPool → Conv2D(128) → Flatten → Dense(256) → Dropout → Output`


In [ ]:
def build_model_B(input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES, small_image_safe=False):
    """
    Build the deeper CNN.

    small_image_safe=False:
        Use the MNIST-friendly version with three pooling operations.
        This works well for 28x28 images: 28 -> 14 -> 7 -> 3.

    small_image_safe=True:
        Use the 9x9-safe version with only one pooling operation.
        This avoids collapsing 9x9 inputs too early: 9 -> 9 -> 9 -> 4.
    """
    if not small_image_safe:
        model = Sequential([
            Input(shape=input_shape, name="input_B"),

            # Layer 1: low-level visual features
            Conv2D(32, kernel_size=(3, 3), activation="relu", padding="same", name="conv_B1"),
            MaxPooling2D(pool_size=(2, 2), name="pool_B1"),

            # Layer 2: mid-level visual combinations
            Conv2D(64, kernel_size=(3, 3), activation="relu", padding="same", name="conv_B2"),
            MaxPooling2D(pool_size=(2, 2), name="pool_B2"),

            # Layer 3: more abstract digit-level combinations
            Conv2D(128, kernel_size=(3, 3), activation="relu", padding="same", name="conv_B3"),
            MaxPooling2D(pool_size=(2, 2), name="pool_B3"),

            Flatten(name="flatten_B"),
            Dense(256, activation="relu", name="dense_B1"),
            Dropout(0.40, name="dropout_B"),
            Dense(num_classes, activation="softmax", name="output_B")
        ], name="Model_B_Deep_MNIST")
    else:
        model = Sequential([
            Input(shape=input_shape, name="input_B"),

            # For 9x9 inputs, preserve spatial resolution in the early layers.
            Conv2D(32, kernel_size=(3, 3), activation="relu", padding="same", name="conv_B1"),
            Conv2D(64, kernel_size=(3, 3), activation="relu", padding="same", name="conv_B2"),
            MaxPooling2D(pool_size=(2, 2), name="pool_B1"),

            # This layer still receives a 4x4 spatial map for 9x9 inputs.
            Conv2D(128, kernel_size=(3, 3), activation="relu", padding="same", name="conv_B3"),

            Flatten(name="flatten_B"),
            Dense(256, activation="relu", name="dense_B1"),
            Dropout(0.40, name="dropout_B"),
            Dense(num_classes, activation="softmax", name="output_B")
        ], name="Model_B_Deep_9x9_Safe")

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

# For the current MNIST baseline, use the MNIST-friendly architecture.
# When switching to 9x9 course data, change small_image_safe=True.
model_B = build_model_B(small_image_safe=False)
model_B.summary()


## Cell 5b — Spatial Dimension Check for MNIST vs. 9×9 Data

This diagnostic cell explains why the current MNIST architecture works for 28×28 images, but must be adapted for 9×9 course images.

The key risk is **spatial collapse**: too many MaxPooling layers can reduce a 9×9 image to 1×1, leaving little spatial structure for deeper convolutional layers to analyze.


In [ ]:
def simulate_pooling_size(input_size, num_pools):
    """Approximate spatial size after repeated 2x2 max-pooling with default valid behavior."""
    size = input_size
    sizes = [size]
    for _ in range(num_pools):
        size = size // 2
        sizes.append(size)
    return sizes

dimension_rows = [
    {
        "Dataset / Architecture": "MNIST 28x28 with 3 pools",
        "Spatial sizes": " -> ".join(map(str, simulate_pooling_size(28, 3))),
        "Interpretation": "OK: final map is still about 3x3"
    },
    {
        "Dataset / Architecture": "9x9 with 3 pools",
        "Spatial sizes": " -> ".join(map(str, simulate_pooling_size(9, 3))),
        "Interpretation": "Risky: collapses to about 1x1"
    },
    {
        "Dataset / Architecture": "9x9-safe Model B with 1 pool",
        "Spatial sizes": "9 -> 9 -> 9 -> 4 -> conv",
        "Interpretation": "Better: preserves spatial structure for conv_B3"
    }
]

dimension_df = pd.DataFrame(dimension_rows)
display(dimension_df)


## Cell 6 — Train Both Models

In [ ]:
NUM_EPOCHS = 10
BATCH_SIZE = 64

# Early stopping helps avoid unnecessary overfitting while keeping the notebook fast.
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True,
    verbose=1
)

def train_and_evaluate(model, model_label):
    print("=" * 70)
    print(f"Training {model_label}...")
    print("=" * 70)

    start = time.time()
    history = model.fit(
        x_train, y_train,
        batch_size=BATCH_SIZE,
        epochs=NUM_EPOCHS,
        validation_split=0.10,
        callbacks=[early_stop],
        verbose=1
    )
    elapsed = round(time.time() - start)

    test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)

    print(f"\n{model_label} training complete.")
    print(f"Training time: {elapsed} seconds")
    print(f"Test loss:     {test_loss:.4f}")
    print(f"Test accuracy: {test_acc:.4f}")

    return history, test_loss, test_acc, elapsed

history_A, loss_A, acc_A, elapsed_A = train_and_evaluate(model_A, "Model A — Shallow CNN")
history_B, loss_B, acc_B, elapsed_B = train_and_evaluate(model_B, "Model B — Deep CNN")

print("\n--- Accuracy Summary ---")
print(f"Model A test accuracy: {acc_A*100:.2f}%")
print(f"Model B test accuracy: {acc_B*100:.2f}%")

## Cell 7 — Training Curves: Accuracy and Loss

In [ ]:
def plot_training_curves(history_A, history_B):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Model A vs Model B — Training Curves", fontsize=13, fontweight="bold")

    # Accuracy
    axes[0].plot(history_A.history["accuracy"], label="A train")
    axes[0].plot(history_A.history["val_accuracy"], linestyle="--", label="A validation")
    axes[0].plot(history_B.history["accuracy"], label="B train")
    axes[0].plot(history_B.history["val_accuracy"], linestyle="--", label="B validation")
    axes[0].set_title("Accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Loss
    axes[1].plot(history_A.history["loss"], label="A train")
    axes[1].plot(history_A.history["val_loss"], linestyle="--", label="A validation")
    axes[1].plot(history_B.history["loss"], label="B train")
    axes[1].plot(history_B.history["val_loss"], linestyle="--", label="B validation")
    axes[1].set_title("Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "training_curves_comparison.png"), dpi=150, bbox_inches="tight")
    plt.show()

plot_training_curves(history_A, history_B)

## Cell 8 — Select Representative Test Images

Using multiple digits avoids over-interpreting the feature maps from only one sample.

In [ ]:
# Representative digits: loop, simple stroke, complex stroke, double-loop structure
DIGITS_TO_VISUALIZE = [0, 1, 5, 8]

representative_indices = []
for digit in DIGITS_TO_VISUALIZE:
    representative_indices.append(np.where(y_test == digit)[0][0])

representative_images = x_test[representative_indices]
representative_labels = y_test[representative_indices]

fig, axes = plt.subplots(1, len(DIGITS_TO_VISUALIZE), figsize=(10, 3))
fig.suptitle("Representative Test Images for Feature-Map Analysis", fontsize=12, fontweight="bold")

for ax, img, label in zip(axes, representative_images, representative_labels):
    ax.imshow(img.reshape(28, 28), cmap="gray")
    ax.set_title(f"Digit {label}")
    ax.axis("off")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "representative_inputs.png"), dpi=150, bbox_inches="tight")
plt.show()

## Cell 9 — Feature Map Visualization Functions

This cell visualizes activations from hidden convolutional layers.  
For the main report, the most useful view is usually the **mean activation map** across channels because it gives one compact image per layer.

In [ ]:
def get_layer_activations(model, layer_name, images):
    """Return activations from a target layer for a batch of images."""
    intermediate_model = Model(
        inputs=model.input,
        outputs=model.get_layer(layer_name).output
    )
    return intermediate_model.predict(images, verbose=0)


def plot_individual_feature_maps(model, layer_name, image, title, n_filters=16, save_name=None):
    """
    Plot selected individual channel activations for one image.
    This is useful for seeing what different filters respond to.
    """
    activations = get_layer_activations(model, layer_name, np.expand_dims(image, axis=0))
    n_filters = min(n_filters, activations.shape[-1])

    cols = 8
    rows = int(np.ceil(n_filters / cols))

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.4, rows * 1.4))
    fig.suptitle(title, fontsize=11, fontweight="bold")
    axes = np.array(axes).reshape(-1)

    for i, ax in enumerate(axes):
        if i < n_filters:
            ax.imshow(activations[0, :, :, i], cmap="viridis")
        ax.axis("off")

    plt.tight_layout()
    if save_name:
        plt.savefig(os.path.join(OUTPUT_DIR, save_name), dpi=150, bbox_inches="tight")
    plt.show()

    return activations


def plot_mean_activation_progression(model, layer_names, images, labels, title, save_name=None):
    """
    Plot one mean activation map per layer for multiple representative images.
    Mean activation = average over all feature channels.
    """
    n_rows = len(images)
    n_cols = len(layer_names) + 1

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.0 * n_cols, 2.8 * n_rows))
    fig.suptitle(title, fontsize=13, fontweight="bold")

    if n_rows == 1:
        axes = np.expand_dims(axes, axis=0)

    for r, (img, label) in enumerate(zip(images, labels)):
        # Input column
        axes[r, 0].imshow(img.reshape(28, 28), cmap="gray")
        axes[r, 0].set_title(f"Input\nDigit {label}")
        axes[r, 0].axis("off")

        for c, layer_name in enumerate(layer_names, start=1):
            acts = get_layer_activations(model, layer_name, np.expand_dims(img, axis=0))
            mean_map = np.mean(acts[0], axis=-1)

            axes[r, c].imshow(mean_map, cmap="viridis")
            axes[r, c].set_title(layer_name)
            axes[r, c].axis("off")

    plt.tight_layout()
    if save_name:
        plt.savefig(os.path.join(OUTPUT_DIR, save_name), dpi=150, bbox_inches="tight")
    plt.show()

## Cell 10 — Model A Feature Maps

In [ ]:
# Model A only has one convolutional layer.
sample_image = representative_images[2]  # digit 5
sample_label = representative_labels[2]

plot_individual_feature_maps(
    model_A,
    layer_name="conv_A1",
    image=sample_image,
    title=f"Model A — Individual Feature Maps from conv_A1 (input digit {sample_label})",
    n_filters=16,
    save_name="featuremaps_A_layer1_individual.png"
)

plot_mean_activation_progression(
    model_A,
    layer_names=["conv_A1"],
    images=representative_images,
    labels=representative_labels,
    title="Model A — Mean Activation Map Progression",
    save_name="featuremaps_A_mean_progression.png"
)

## Cell 11 — Model B Feature Maps Across Depth

In [ ]:
# Individual feature maps for one sample at each layer
for layer_name, layer_description in [
    ("conv_B1", "Layer 1: edges / simple strokes"),
    ("conv_B2", "Layer 2: corners / shape fragments"),
    ("conv_B3", "Layer 3: more abstract digit combinations")
]:
    plot_individual_feature_maps(
        model_B,
        layer_name=layer_name,
        image=sample_image,
        title=f"Model B — {layer_description} ({layer_name}), input digit {sample_label}",
        n_filters=16,
        save_name=f"featuremaps_B_{layer_name}_individual.png"
    )

# Mean activation progression across multiple representative digits
plot_mean_activation_progression(
    model_B,
    layer_names=["conv_B1", "conv_B2", "conv_B3"],
    images=representative_images,
    labels=representative_labels,
    title="Model B — Mean Activation Progression Across CNN Depth",
    save_name="featuremaps_B_mean_progression.png"
)

## Cell 12 — Learned Filter Visualization

Important interpretation note:  
- First-layer filters are directly interpretable because the input has one grayscale channel.  
- Deeper filters have many input channels, so the code averages absolute filter weights across input channels rather than showing only channel 0.

In [ ]:
def visualize_filters(model, layer_name, title, save_name=None):
    """
    Visualize learned convolutional filter weights.
    For deeper layers, average absolute filter weights across input channels.
    """
    filters, biases = model.get_layer(layer_name).get_weights()
    # Filter shape: (kernel_height, kernel_width, input_channels, number_of_filters)

    if filters.shape[2] > 1:
        filter_maps = np.mean(np.abs(filters), axis=2)
    else:
        filter_maps = filters[:, :, 0, :]

    # Normalize for display
    f_min, f_max = filter_maps.min(), filter_maps.max()
    filter_maps = (filter_maps - f_min) / (f_max - f_min + 1e-8)

    n_filters = min(16, filter_maps.shape[-1])
    cols = 8
    rows = int(np.ceil(n_filters / cols))

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.4, rows * 1.4))
    fig.suptitle(title, fontsize=11, fontweight="bold")
    axes = np.array(axes).reshape(-1)

    for i, ax in enumerate(axes):
        if i < n_filters:
            ax.imshow(filter_maps[:, :, i], cmap="gray")
        ax.axis("off")

    plt.tight_layout()
    if save_name:
        plt.savefig(os.path.join(OUTPUT_DIR, save_name), dpi=150, bbox_inches="tight")
    plt.show()


visualize_filters(
    model_A,
    "conv_A1",
    "Model A — Learned Filters in conv_A1",
    save_name="filters_A_conv_A1.png"
)

for layer_name in ["conv_B1", "conv_B2", "conv_B3"]:
    visualize_filters(
        model_B,
        layer_name,
        f"Model B — Learned Filters in {layer_name}",
        save_name=f"filters_B_{layer_name}.png"
    )

## Cell 13 — Confusion Matrices

In [ ]:
preds_A = np.argmax(model_A.predict(x_test, verbose=0), axis=1)
preds_B = np.argmax(model_B.predict(x_test, verbose=0), axis=1)

def plot_confusion_matrix(y_true, y_pred, title, ax):
    cm = pd.crosstab(
        pd.Series(y_true, name="Actual"),
        pd.Series(y_pred, name="Predicted")
    )

    # Ensure all labels 0-9 appear even if a prediction class is absent.
    cm = cm.reindex(index=range(10), columns=range(10), fill_value=0)

    cm_errors = cm.copy()
    np.fill_diagonal(cm_errors.values, 0)

    sns.heatmap(
        cm_errors,
        cmap="Reds",
        annot=True,
        fmt=".0f",
        linewidths=0.5,
        annot_kws={"size": 8},
        cbar=False,
        square=True,
        ax=ax
    )

    accuracy = np.trace(cm.values) / np.sum(cm.values)
    ax.set_title(f"{title}\nErrors only, accuracy={accuracy*100:.2f}%", fontweight="bold")
    ax.set_xlabel("Predicted digit")
    ax.set_ylabel("Actual digit")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Confusion Matrices — Misclassifications Only", fontsize=13, fontweight="bold")

plot_confusion_matrix(y_test, preds_A, "Model A — Shallow CNN", axes[0])
plot_confusion_matrix(y_test, preds_B, "Model B — Deep CNN", axes[1])

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrices_errors_only.png"), dpi=150, bbox_inches="tight")
plt.show()

## Cell 14 — Activation Statistics

This measures how dense or sparse each layer's activations are.  
Higher sparsity often suggests more selective representations, but it should be interpreted together with feature maps and PCA results.

In [ ]:
def compute_activation_stats(model, layer_name, images, n_samples=500):
    intermediate_model = Model(
        inputs=model.input,
        outputs=model.get_layer(layer_name).output
    )
    activations = intermediate_model.predict(images[:n_samples], verbose=0)

    mean_activation = np.mean(activations)
    median_activation = np.median(activations)
    sparsity = np.mean(activations == 0)  # ReLU zeros

    return {
        "Layer": layer_name,
        "Mean activation": mean_activation,
        "Median activation": median_activation,
        "Sparsity": sparsity
    }

layers_to_check = [
    (model_A, "conv_A1", "A: Conv1"),
    (model_B, "conv_B1", "B: Conv1"),
    (model_B, "conv_B2", "B: Conv2"),
    (model_B, "conv_B3", "B: Conv3"),
]

stats_rows = []
for model, layer_name, label in layers_to_check:
    row = compute_activation_stats(model, layer_name, x_test, n_samples=500)
    row["Label"] = label
    stats_rows.append(row)

stats_df = pd.DataFrame(stats_rows)[["Label", "Layer", "Mean activation", "Median activation", "Sparsity"]]
stats_df["Sparsity (%)"] = stats_df["Sparsity"] * 100

display(stats_df)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Activation Statistics by Layer", fontsize=12, fontweight="bold")

axes[0].bar(stats_df["Label"], stats_df["Mean activation"])
axes[0].set_title("Mean Activation")
axes[0].set_ylabel("Mean activation value")
axes[0].grid(axis="y", alpha=0.3)

axes[1].bar(stats_df["Label"], stats_df["Sparsity (%)"])
axes[1].set_title("Activation Sparsity")
axes[1].set_ylabel("Percent zeros after ReLU")
axes[1].grid(axis="y", alpha=0.3)

for ax in axes:
    ax.tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "activation_statistics.png"), dpi=150, bbox_inches="tight")
plt.show()

## Cell 15 — PCA + Silhouette Score

This is the strongest quantitative evidence in the notebook.

- **2D PCA** is used for visualization.  
- **Higher-dimensional PCA** is used for silhouette score, so the metric is not overly dependent on a 2D projection.  
- To avoid avoidable leakage, the scaler and PCA transforms are **fit on training representations** and then applied to test representations.

A higher silhouette score means the digit classes are more separated in the representation space.


In [ ]:
def get_layer_representation(model, layer_name, images, n_samples=800):
    """Extract flattened intermediate-layer activations."""
    intermediate_model = Model(
        inputs=model.input,
        outputs=model.get_layer(layer_name).output
    )
    activations = intermediate_model.predict(images[:n_samples], verbose=0)
    return activations.reshape(n_samples, -1)


def pca_fit_train_transform_test(X_train_rep, X_test_rep, y_test_sample, max_components_for_score=20):
    """
    Fit preprocessing and PCA on training representations, then transform test representations.

    Returns:
    - 2D PCA coordinates for test-set visualization
    - silhouette score computed on a higher-dimensional PCA representation of test data
    """
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_rep)
    X_test_scaled = scaler.transform(X_test_rep)

    # 2D PCA for plotting
    pca_2d = PCA(n_components=2, random_state=SEED, svd_solver="randomized")
    pca_2d.fit(X_train_scaled)
    X_test_2d = pca_2d.transform(X_test_scaled)

    # Higher-dimensional PCA for silhouette score
    n_components = min(max_components_for_score, X_train_scaled.shape[1], X_train_scaled.shape[0] - 1)
    pca_high = PCA(n_components=n_components, random_state=SEED, svd_solver="randomized")
    pca_high.fit(X_train_scaled)
    X_test_high = pca_high.transform(X_test_scaled)

    score = silhouette_score(X_test_high, y_test_sample)
    return X_test_2d, score


N_REP_SAMPLES = 800
y_sample = y_test[:N_REP_SAMPLES]

# Build train/test representation pairs.
# Raw pixels are included as the baseline representation before the CNN has transformed the image.
representation_pairs = {
    "Raw pixels": (
        x_train[:N_REP_SAMPLES].reshape(N_REP_SAMPLES, -1),
        x_test[:N_REP_SAMPLES].reshape(N_REP_SAMPLES, -1)
    ),
    "Conv B1": (
        get_layer_representation(model_B, "conv_B1", x_train, N_REP_SAMPLES),
        get_layer_representation(model_B, "conv_B1", x_test, N_REP_SAMPLES)
    ),
    "Conv B2": (
        get_layer_representation(model_B, "conv_B2", x_train, N_REP_SAMPLES),
        get_layer_representation(model_B, "conv_B2", x_test, N_REP_SAMPLES)
    ),
    "Conv B3": (
        get_layer_representation(model_B, "conv_B3", x_train, N_REP_SAMPLES),
        get_layer_representation(model_B, "conv_B3", x_test, N_REP_SAMPLES)
    ),
    "Dense B1": (
        get_layer_representation(model_B, "dense_B1", x_train, N_REP_SAMPLES),
        get_layer_representation(model_B, "dense_B1", x_test, N_REP_SAMPLES)
    ),
}

pca_coords = {}
silhouette_results = []

for name, (X_train_rep, X_test_rep) in representation_pairs.items():
    X_2d, score = pca_fit_train_transform_test(X_train_rep, X_test_rep, y_sample)
    pca_coords[name] = X_2d
    silhouette_results.append({"Representation": name, "Silhouette score": score})
    print(f"{name:<12}: silhouette = {score:.4f}")

# Plot PCA progression
fig, axes = plt.subplots(1, len(pca_coords), figsize=(20, 4))
fig.suptitle("PCA of Representations Across CNN Depth", fontsize=13, fontweight="bold")

for ax, (name, X_2d) in zip(axes, pca_coords.items()):
    sc = ax.scatter(
        X_2d[:, 0], X_2d[:, 1],
        c=y_sample,
        cmap="tab10",
        s=8,
        alpha=0.70
    )
    score = [row["Silhouette score"] for row in silhouette_results if row["Representation"] == name][0]
    ax.set_title(f"{name}\nSilhouette={score:.3f}", fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "pca_representations_by_layer.png"), dpi=150, bbox_inches="tight")
plt.show()

# Silhouette score bar chart
silhouette_df = pd.DataFrame(silhouette_results)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=silhouette_df, x="Representation", y="Silhouette score", ax=ax)
ax.set_title("Class Separability by Representation", fontweight="bold")
ax.set_xlabel("Representation")
ax.set_ylabel("Silhouette score")
ax.tick_params(axis="x", rotation=25)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "silhouette_scores_by_layer.png"), dpi=150, bbox_inches="tight")
plt.show()

display(silhouette_df)


## Cell 16 — Final Results Summary

In [ ]:
print("=" * 75)
print("FINAL RESULTS SUMMARY")
print("=" * 75)

total_A = model_A.count_params()
total_B = model_B.count_params()

summary_rows = [
    {
        "Model": "Model A — Shallow CNN",
        "Conv layers": 1,
        "Parameters": total_A,
        "Test accuracy": acc_A,
        "Test loss": loss_A,
        "Training time (sec)": elapsed_A
    },
    {
        "Model": "Model B — Deep CNN",
        "Conv layers": 3,
        "Parameters": total_B,
        "Test accuracy": acc_B,
        "Test loss": loss_B,
        "Training time (sec)": elapsed_B
    }
]

summary_df = pd.DataFrame(summary_rows)
summary_df["Test accuracy (%)"] = summary_df["Test accuracy"] * 100
display(summary_df[["Model", "Conv layers", "Parameters", "Test accuracy (%)", "Test loss", "Training time (sec)"]])

print("\nKey interpretation:")
print(
    "The deeper CNN appears to learn more selective and class-separable internal representations. "
    "The evidence comes from feature maps, activation sparsity, PCA visualization, and silhouette scores. "
    "However, first-layer filters are much easier to interpret directly than deeper filters, because deeper "
    "filters operate on prior feature maps rather than raw pixels."
)

print("\n9x9 extension note:")
print(
    "If this notebook is adapted to the course-provided 9x9 letters/shapes/lines dataset, use "
    "build_model_B(..., small_image_safe=True). This avoids collapsing spatial dimensions too early with repeated MaxPooling."
)

print("\nImportant limitation:")
print(
    "This is a preliminary MNIST baseline. For the final report, the same workflow can be extended "
    "to the course-provided visual data or to another image dataset if time allows."
)


## Appendix — Optional 9×9 Course-Data Adaptation Template

Use this cell later **only after** the 9×9 course dataset has been converted into arrays shaped like:

`x_train_9x9.shape = (N, 9, 9, 1)`  
`y_train_9x9.shape = (N,)`

The important change is `small_image_safe=True`, which reduces pooling so the spatial representation does not collapse too early.


In [ ]:
# Example only — run this after you prepare the 9x9 dataset arrays.
# Do not run until x_train_9x9, y_train_9x9, x_test_9x9, and y_test_9x9 exist.

# INPUT_SHAPE_9X9 = (9, 9, 1)
# NUM_CLASSES_9X9 = len(np.unique(y_train_9x9))

# model_A_9x9 = build_model_A(input_shape=INPUT_SHAPE_9X9, num_classes=NUM_CLASSES_9X9)
# model_B_9x9 = build_model_B(input_shape=INPUT_SHAPE_9X9, num_classes=NUM_CLASSES_9X9, small_image_safe=True)

# model_A_9x9.summary()
# model_B_9x9.summary()
